# Jacobian lens on Gemma 4 E4B — text-only pilot

**Research question.** Does the Jacobian lens produce stable, interpretable, and
meaningfully transported vocabulary readouts from intermediate residual states
in `google/gemma-4-E4B-it`?

**Method source (authoritative):** [Verbalizable Representations Form a Global
Workspace in Language Models](https://transformer-circuits.pub/2026/workspace/index.html)
(Anthropic, 2026).
**Implementation scaffold:** the official reference implementation
[anthropics/jacobian-lens](https://github.com/anthropics/jacobian-lens)
(Apache-2.0), used here as the `upstream` git remote. This repository adapts it
to Gemma 4 with a narrow adapter (`jlens/gemma4.py`), controls
(`jlens/controls.py`), and metadata (`jlens/metadata.py`); the upstream
`jlens` core is unmodified.

**Jacobian convention** (paper §, upstream `jlens/fitting.py`):

$$J_\ell = \mathbb{E}_{\text{prompts},\,t,\,t' \ge t}\left[\frac{\partial h_{\text{final},t'}}{\partial h_{\ell,t}}\right],\qquad \text{lens}_\ell(h) = \mathrm{softmax}(W_U\,\mathrm{norm}(J_\ell h))$$

- `J[i, j] = ∂ h_final[dim i] / ∂ h_l[dim j]` — **rows are target dims, columns
  are source dims**; shape `[d_model, d_model] = [2560, 2560]`, fp32.
- Forward transport is `J @ h`, implemented as `residual @ J.T`
  (`JacobianLens.transport`). No other transposes appear in the pipeline.
- Estimator: one-hot cotangents at **every valid target position at once**;
  causal masking makes `t' < t` contributions exactly zero, so the gradient at
  source position `t` is the sum over `t' ≥ t`, then averaged over source
  positions (positions `0..15` and the final position are excluded) and prompts.
- Source site: output of `model.language_model.layers[l]` (after attention,
  MLP, per-layer-embedding re-injection, `layer_scalar`) — the input to block
  `l+1`. Target site: the same at block 41 (pre-final-norm residual).
- Readout: final RMSNorm → tied unembedding `[262144, 2560]` → **pre-softcap
  logits** (paper convention) and `30·tanh(x/30)` **softcapped logits**
  (Gemma's actual output pathway). The cap is monotonic ⇒ identical rankings.

Orientation is pinned by tests (`tests/test_fitting.py::test_jacobian_for_prompt_tiny`,
`tests/test_gemma4_adapter.py::test_fit_and_apply_through_adapter`,
`tests/test_finite_difference.py`). **Do not proceed if any of them fail.**


In [ ]:
# 1. Environment, upstream commit, repository provenance (no model load).
import json, os, sys, pathlib

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "jlens").exists():          # notebook started inside notebooks/
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from jlens.metadata import environment_manifest, UPSTREAM_COMMIT, UPSTREAM_REPO_URL

ENV = environment_manifest()
print(f"upstream: {UPSTREAM_REPO_URL} @ {UPSTREAM_COMMIT}")
print(json.dumps(ENV, indent=2))
IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)


### Configuration and gating

`MODE` selects `configs/gemma_text_{microsmoke,smoke,pilot}.yaml`.
Real Gemma execution (~16 GB download + load) happens **only** if
`ALLOW_MODEL_LOAD` is true — set the environment variable `JLENS_ALLOW_GEMMA=1`
(or flip the constant) deliberately. On Colab use an **A100** runtime; begin
with the config's small `dim_batch` and scale up only after the memory probe.
Stage order: `microsmoke` (1–2 prompts, one middle layer, seq ≤ 48,
dim_batch 4) → `smoke` (8 prompts, 5 layers) → `pilot` (100 WikiText
sequences, 7 layers). Smoke-mode lens quality does **not** represent the
method; only the pilot is worth interpreting.


In [ ]:
# 2. Load and validate the experiment configuration.
from jlens.metadata import load_config, config_fingerprint

MODE = os.environ.get("JLENS_MODE", "microsmoke")            # microsmoke | smoke | pilot
ALLOW_MODEL_LOAD = os.environ.get("JLENS_ALLOW_GEMMA", "0") == "1"
DEVICE_MAP = os.environ.get("JLENS_DEVICE_MAP") or None       # e.g. "cuda" on the A100

CONFIG_PATH = f"configs/gemma_text_{MODE}.yaml"
config = load_config(CONFIG_PATH)
FINGERPRINT = config_fingerprint(config)
print(f"config: {CONFIG_PATH}\nfingerprint: {FINGERPRINT}")
print(f"mode={config['mode']}  source_layers={config['sites']['source_layers']}  "
      f"target_layer={config['sites']['target_layer']}")
print(f"ALLOW_MODEL_LOAD={ALLOW_MODEL_LOAD}  DEVICE_MAP={DEVICE_MAP}")
OUTPUT_DIR = pathlib.Path(config["paths"]["output_dir"]); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 3. Lightweight validation suite (CPU, mocks, no network) — must pass before
# any real-model work.
import subprocess
proc = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--no-header"],
                      capture_output=True, text=True, cwd=REPO_ROOT)
print(proc.stdout[-2000:])
assert proc.returncode == 0, "local validation suite failed — do not proceed"


In [ ]:
# 4. Resolve the immutable model revision (network, but no model download).
from jlens.gemma4 import resolve_revision

MODEL_REVISION = None
if ALLOW_MODEL_LOAD:
    MODEL_REVISION = resolve_revision(config["model"]["repo_id"],
                                      config["model"]["revision"])
    print(f"{config['model']['repo_id']} pinned to {MODEL_REVISION}")
else:
    print("model load disabled — skipping revision resolution")


In [ ]:
# 5. Load Gemma 4 (gated) and verify the architecture.
model = None
if ALLOW_MODEL_LOAD:
    import torch
    from jlens.gemma4 import load_gemma4, verify_architecture

    dtype = {"bfloat16": torch.bfloat16, "float32": torch.float32}[config["model"]["dtype"]]
    model, LOAD_INFO = load_gemma4(
        config["model"]["repo_id"], revision=MODEL_REVISION, dtype=dtype,
        device_map=DEVICE_MAP, allow_model_load=True)
    ARCH_REPORT = verify_architecture(
        model,
        expect_n_layers=config["model"]["expect_n_layers"],
        expect_d_model=config["model"]["expect_d_model"],
        expect_vocab_size=config["model"]["expect_vocab_size"])
    print(json.dumps({**LOAD_INFO, **{k: v for k, v in ARCH_REPORT.to_dict().items()
                                      if k != "layer_scalars"}}, indent=2, default=str))
    print("layer_scalars all unit:", ARCH_REPORT.layer_scalars_all_unit)
else:
    print("model load disabled — skipping (set JLENS_ALLOW_GEMMA=1)")


In [ ]:
# 6. Hook-site resolution + memory/runtime probe at the configured dim_batch.
# Scale dim_batch upward ONLY after inspecting the probe's peak memory.
PROBE = None
if model is not None:
    from jlens.gemma4 import probe_fit_cost
    import json as _json

    print("source blocks:", [f"model.language_model.layers[{l}]"
                             for l in config["sites"]["source_layers"]])
    print(f"target block:  model.language_model.layers[{config['sites']['target_layer']}]")
    fit_prompts_preview = json.load(open(config["fitting"]["prompts_path"] or
                                         "configs/prompts/fit_prompts.json",
                                         encoding="utf-8"))["prompts"]
    PROBE = probe_fit_cost(model, fit_prompts_preview[0],
                           config["sites"]["source_layers"],
                           dim_batch=config["fitting"]["dim_batch"],
                           max_seq_len=min(48, config["fitting"]["max_seq_len"]))
    print(_json.dumps(PROBE, indent=2))
    assert PROBE["all_finite"]


In [ ]:
# 7. Fit the lens (checkpointed, resumable) — or load an existing artifact.
LENS = None
if model is not None:
    import time
    from jlens.fitting import fit
    from jlens.lens import JacobianLens
    from jlens.metadata import environment_manifest, prompt_hashes, write_metadata

    lens_path = OUTPUT_DIR / "lens.pt"
    if lens_path.exists():
        LENS = JacobianLens.load(str(lens_path))
        print("loaded existing", LENS)
    else:
        if config["fitting"]["prompt_source"] == "file":
            prompts = json.load(open(config["fitting"]["prompts_path"],
                                     encoding="utf-8"))["prompts"][:config["fitting"]["n_prompts"]]
        else:
            from jlens.examples import load_wikitext_prompts
            prompts = load_wikitext_prompts(n_prompts=config["fitting"]["n_prompts"])
        start = time.perf_counter()
        LENS = fit(model, prompts,
                   source_layers=config["sites"]["source_layers"],
                   target_layer=config["sites"]["target_layer"],
                   dim_batch=config["fitting"]["dim_batch"],
                   max_seq_len=config["fitting"]["max_seq_len"],
                   skip_first=config["positions"]["skip_first"],
                   checkpoint_path=config["paths"]["checkpoint"],
                   checkpoint_every=config["fitting"]["checkpoint_every"])
        runtime = time.perf_counter() - start
        LENS.save(str(lens_path))
        assert JacobianLens.load(str(lens_path)).source_layers == LENS.source_layers
        write_metadata(str(OUTPUT_DIR / "fit_metadata.json"), {
            "config": config, "config_fingerprint": FINGERPRINT,
            "load_info": LOAD_INFO, "architecture_report": ARCH_REPORT.to_dict(),
            "probe": PROBE, "prompt_hashes": prompt_hashes(prompts),
            "n_prompts_fitted": LENS.n_prompts,
            "fit_runtime_seconds": round(runtime, 1),
            "environment": environment_manifest()})
        print(f"fitted + saved {lens_path} in {runtime:.0f}s:", LENS)


In [ ]:
# 8. Logit lens vs J-lens on the evaluation prompts.
# Plain-text and chat-templated prompts are evaluated SEPARATELY; the fitting
# corpus was plain text only. Both pre-softcap (paper convention) and
# softcapped (Gemma pathway) logit values are shown; rankings are identical.
if LENS is not None:
    from jlens.gemma4 import apply_dual

    eval_payload = json.load(open(config["eval"]["prompts_path"], encoding="utf-8"))
    K = config["eval"]["top_k"]

    def show(text, slug, fmt, positions):
        lens_logits, model_logits, ids = apply_dual(LENS, model, text, positions=positions)
        logit_logits, _, _ = apply_dual(LENS, model, text, positions=positions,
                                        use_jacobian=False)
        print(f"\n=== {slug} [{fmt}] (seq_len={ids.shape[1]}) ===")
        for i, pos in enumerate(positions):
            actual = model.tokenizer.decode([int(model_logits["pre"][i].argmax())])
            print(f" position {pos} — model top-1: {actual!r}")
            for layer in LENS.source_layers:
                jl = lens_logits[layer]["pre"][i]; jc = lens_logits[layer]["capped"][i]
                ll = logit_logits[layer]["pre"][i]
                fmt_top = lambda t, n=5: " ".join(repr(model.tokenizer.decode([int(x)]))
                                                  for x in t.topk(n).indices)
                print(f"  L{layer:>2}  J-lens: {fmt_top(jl)}")
                print(f"       (top-1 logit pre={jl.max():.2f} capped={jc.max():.2f})")
                print(f"       logit-lens: {fmt_top(ll)}")

    for entry in eval_payload["plain"]:
        show(entry["text"], entry["slug"], "plain", entry.get("positions", [-1]))
    for entry in eval_payload["chat"]:
        messages = ([{"role": "system", "content": entry["system"]}] if entry.get("system") else [])
        messages.append({"role": "user", "content": entry["user"]})
        text = model.tokenizer.apply_chat_template(messages, tokenize=False,
                                                   add_generation_prompt=True)
        show(text, entry["slug"], "chat", entry.get("positions", [-1]))


In [ ]:
# 9. Negative controls. Primary: row-permuted fitted J (same entries, transport
# destroyed). Also: scale-matched random matrix, wrong-layer application, and
# the identity/logit lens. Metrics vs the model's real output distribution.
if LENS is not None:
    from jlens.controls import control_lens, wrong_layer_lens, topk_overlap, ranks_of_targets

    controls = {"permuted": control_lens(LENS, "permuted", seed=config["eval"]["control_seed"]),
                "random": control_lens(LENS, "random", seed=config["eval"]["control_seed"])}
    if len(LENS.source_layers) >= 2:
        controls["wrong_layer"] = wrong_layer_lens(LENS)

    text = eval_payload["plain"][0]["text"]; positions = [-2, -1]
    jl, mdl, _ = apply_dual(LENS, model, text, positions=positions)
    ll, _, _ = apply_dual(LENS, model, text, positions=positions, use_jacobian=False)
    model_top1 = mdl["pre"].argmax(-1)
    CONTROL_ROWS = []
    print(f"top-{config['eval']['top_k']} overlap with model output / rank of model top-1")
    print(f"{'layer':>6} {'J-lens':>14} {'logit-lens':>14} {'permuted':>14} {'random':>14} {'wrong-layer':>14}")
    for layer in LENS.source_layers:
        row = {"layer": layer}
        variants = {"J-lens": jl[layer]["pre"], "logit-lens": ll[layer]["pre"]}
        for name, c in controls.items():
            variants[name] = c.apply(model, text, layers=[layer], positions=positions)[0][layer].float()
        cols = []
        for name, logits in variants.items():
            ov = topk_overlap(logits, mdl["pre"], config["eval"]["top_k"])
            rk = int(ranks_of_targets(logits, model_top1)[-1])
            row[name] = {"overlap": round(ov, 3), "rank_of_model_top1": rk}
            cols.append(f"{ov:.2f} / r{rk:>5}")
        CONTROL_ROWS.append(row)
        print(f"{layer:>6} " + " ".join(f"{c:>14}" for c in cols))
    print("\nMeaningful transport ⇒ J-lens should beat permuted/random/wrong-layer,"
          "\nespecially at early/middle layers where the logit lens also degrades.")


In [ ]:
# 10. Optional: upstream slice visualisation (layer × position heatmap).
# Reused as-is from upstream jlens.vis; skipped gracefully if incompatible.
if LENS is not None:
    try:
        from jlens.vis import build_page, compute_slice, notebook_iframe
        slice_prompt = eval_payload["plain"][0]["text"]
        slice_data = compute_slice(model, LENS, slice_prompt)
        page, _, _ = build_page(slice_data, slice_prompt,
                                title="Gemma 4 E4B — multihop-currency",
                                description="J-lens slice (upstream visualisation)",
                                mode="embed")
        out = OUTPUT_DIR / "slice_multihop.html"
        out.write_text(page, encoding="utf-8")
        print("wrote", out)
        display(notebook_iframe(page))
    except Exception as exc:                      # noqa: BLE001
        print(f"slice visualisation skipped: {type(exc).__name__}: {exc}")


In [ ]:
# 11. Save the evaluation + control artifacts (configuration-bound).
if LENS is not None:
    from jlens.metadata import write_metadata, environment_manifest
    write_metadata(str(OUTPUT_DIR / "eval_metadata.json"), {
        "config": config, "config_fingerprint": FINGERPRINT,
        "model_revision": MODEL_REVISION,
        "control_rows": CONTROL_ROWS,
        "notes": ("pre-softcap logits follow the paper's lens definition; "
                  "softcapped logits follow Gemma's output pathway; rankings "
                  "identical (monotonic cap). Chat prompts evaluated separately "
                  "from the plain-text fitting distribution."),
        "environment": environment_manifest()})
    print("wrote", OUTPUT_DIR / "eval_metadata.json")
    print(sorted(p.name for p in OUTPUT_DIR.iterdir()))


## Summary and limitations

Fill in after a real run. Template:

- **Stage reached:** microsmoke / smoke / pilot; model revision `…`; runtime, peak memory.
- **Feasibility:** did fitting run end-to-end with finite Jacobians and stable
  per-prompt norms (`fit()` logs `max||J||/sqrt(d)` and running-mean shift)?
- **Interpretability:** at which layers does the J-lens read out
  contextually sensible tokens where the logit lens does not?
- **Controls:** J-lens vs permuted / random / wrong-layer overlap and ranks —
  is apparent interpretability attributable to learned transport?
- **Known limitations:** smoke-scale corpus (≪ paper's ~1000 sequences); bf16
  backward noise; sliding-window + KV-shared attention differ from the paper's
  models; text-only; `-it` chat distribution vs plain-text fitting corpus.

**Extension points (deliberately not implemented):** sparse J-space
decomposition, gradient pursuit, k-cone discovery/visualisation, steering,
multimodal inputs — see README § Roadmap.
